# 01 - Exploratory Data Analysis & Dataset Audit
## Heart Disease Screening and Condition-Risk Prediction System

> **Clinical Decision-Support Prototype Note**:
> This notebook conducts a comprehensive, non-destructive audit and exploratory analysis of the three cardiovascular datasets:
> 1. **Heart Disease (UCI)** - Coronary Artery Disease detection and severity
> 2. **Heart Failure Clinical Records** - Follow-up survival/mortality risk analysis
> 3. **Arrhythmia Conditions** - Multiclass electrophysiological categorization from 12-lead ECG
>
> **Mandatory Clinical & Architectural Principles Applied**:
> - Raw data files are preserved strictly unchanged.
> - Separate dataset targets: `DEATH_EVENT` is analyzed as recorded follow-up mortality in heart failure patients, NOT a diagnosis of heart failure.
> - Data leakage risks (such as follow-up `time` in heart failure or site-specific testing variations in UCI) are identified and evaluated.
> - Arrhythmia input requirements (279 ECG waveform metrics) are evaluated against outpatient screening capabilities to prevent unsupported claims.


In [1]:
import sys
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
PROJECT_ROOT = Path('.').resolve().parent.parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Machine_Learning.src.data_loader import (
    load_heart_disease_data,
    load_heart_failure_data,
    load_arrhythmia_data,
    audit_all_datasets
)

sns.set_theme(style='whitegrid', palette='muted')
print('Environment initialized successfully. Project root:', PROJECT_ROOT)


Environment initialized successfully. Project root: C:\Users\lodha\Downloads\Heart_Disease_Screening


## 1. Unified Dataset Audit
Running non-destructive statistical audit across all three datasets using `Machine_Learning.src.data_loader.audit_all_datasets()`.


In [2]:
raw_data_dir = PROJECT_ROOT / 'Machine_Learning' / 'datasets' / 'raw'
audit_results = audit_all_datasets(raw_data_dir)
audit_summary = []

for name, report in audit_results.items():
    audit_summary.append({
        'Dataset': name,
        'Rows': report.row_count,
        'Columns': report.column_count,
        'Duplicate Rows': report.duplicate_rows,
        'Target': report.target_column,
        'Distinct Target Values': len(report.target_distribution) if report.target_distribution else 0,
        'Features with Missing Data': len(report.missing_values),
        'Flagged Anomalies': len(report.anomalies)
    })

audit_df = pd.DataFrame(audit_summary)
print(audit_df.to_string(index=False))


2026-10-14 23:38:12,239 - Machine_Learning.src.data_loader - INFO - Loaded Heart Disease UCI dataset: shape=(920, 16) from C:\Users\lodha\Downloads\Heart_Disease_Screening\Machine_Learning\datasets\raw\heart_disease_uci.csv


2026-10-14 23:38:12,245 - Machine_Learning.src.data_loader - INFO - Loaded Heart Failure dataset: shape=(299, 13) from C:\Users\lodha\Downloads\Heart_Disease_Screening\Machine_Learning\datasets\raw\heart_failure.csv


2026-10-14 23:38:12,280 - Machine_Learning.src.data_loader - INFO - Loaded Arrhythmia dataset: shape=(430, 280) from C:\Users\lodha\Downloads\Heart_Disease_Screening\Machine_Learning\datasets\raw\arrhythmia_conditions.csv


              Dataset  Rows  Columns  Duplicate Rows          Target  Distinct Target Values  Features with Missing Data  Flagged Anomalies
    heart_disease_uci   920       16               0             num                       5                          10                  2
        heart_failure   299       13               0     DEATH_EVENT                       2                           0                  2
arrhythmia_conditions   430      280               0 condition_group                       5                           5                  2


## 2. Heart Disease UCI Dataset
### 2.1 Overview & Schema Structure
- **Source**: UCI Machine Learning Repository (Cleveland, Hungary, Switzerland, VA Long Beach).
- **Target**: `num` (0: <50% diameter stenosis; 1 to 4: >=50% diameter stenosis in 1 to 4 major coronary arteries).
- **Clinical Task**: Stratification of coronary artery disease presence and severity.


In [3]:
df_hd = load_heart_disease_data()
print('Shape:', df_hd.shape)
print('\nData Types and Null Counts:')
for col in df_hd.columns:
    print(f'  {col:<10}: {str(df_hd[col].dtype):<10} nulls={df_hd[col].isnull().sum()}')
df_hd.head()


2026-10-14 23:38:12,402 - Machine_Learning.src.data_loader - INFO - Loaded Heart Disease UCI dataset: shape=(920, 16) from C:\Users\lodha\Downloads\Heart_Disease_Screening\Machine_Learning\datasets\raw\heart_disease_uci.csv


Shape: (920, 16)

Data Types and Null Counts:
  id        : int64      nulls=0
  age       : int64      nulls=0
  sex       : str        nulls=0
  dataset   : str        nulls=0
  cp        : str        nulls=0
  trestbps  : float64    nulls=59
  chol      : float64    nulls=30
  fbs       : object     nulls=90
  restecg   : str        nulls=2
  thalch    : float64    nulls=55
  exang     : object     nulls=55
  oldpeak   : float64    nulls=62


  slope     : str        nulls=309
  ca        : float64    nulls=611
  thal      : str        nulls=486
  num       : int64      nulls=0


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


### 2.2 Missing Data & Physiological Zero-Value Scan
In clinical research data, unmeasured lab values are occasionally encoded as 0. Here we scan for explicit `NaN` values and biologically impossible zeros.


In [4]:
missing_counts = df_hd.isnull().sum()
missing_table = pd.DataFrame({
    'Missing Count': missing_counts[missing_counts > 0],
    'Percentage (%)': (missing_counts[missing_counts > 0] / len(df_hd) * 100).round(2)
}).sort_values(by='Missing Count', ascending=False)

print('Explicit Missing Values (NaN):')
print(missing_table)

zero_chol = int((df_hd['chol'] == 0).sum())
zero_trestbps = int((df_hd['trestbps'] == 0).sum())
print(f'\nImplicit Missing Data (Physiologically impossible zeros):')
print(f'- Serum Cholesterol == 0: {zero_chol} records ({round(zero_chol / len(df_hd) * 100, 2)}%)')
print(f'- Resting Blood Pressure == 0: {zero_trestbps} records ({round(zero_trestbps / len(df_hd) * 100, 2)}%)')


Explicit Missing Values (NaN):
          Missing Count  Percentage (%)
ca                  611           66.41
thal                486           52.83
slope               309           33.59
fbs                  90            9.78
oldpeak              62            6.74
trestbps             59            6.41
exang                55            5.98
thalch               55            5.98
chol                 30            3.26
restecg               2            0.22

Implicit Missing Data (Physiologically impossible zeros):
- Serum Cholesterol == 0: 172 records (18.7%)
- Resting Blood Pressure == 0: 1 records (0.11%)


### 2.3 Target Class Distribution (`num`)
The target `num` represents the degree of coronary angiographic stenosis (0 = normal, 1-4 = diseased).


In [5]:
num_counts = df_hd['num'].value_counts().sort_index()
binary_cad = (df_hd['num'] > 0).astype(int).value_counts().rename({0: 'No CAD (num=0)', 1: 'CAD Present (num>=1)'})

print('Multiclass distribution (num 0 to 4):')
print(num_counts)
print('\nBinary CAD distribution (Presence vs Absence):')
print(binary_cad)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
num_counts.plot(kind='bar', ax=axes[0], color='cornflowerblue', edgecolor='black')
axes[0].set_title('Heart Disease: Multiclass Severity (num: 0-4)')
axes[0].set_xlabel('Angiographic Severity Stage')
axes[0].set_ylabel('Patient Count')

binary_cad.plot(kind='bar', ax=axes[1], color=['mediumseagreen', 'coral'], edgecolor='black')
axes[1].set_title('Heart Disease: Binary CAD Classification')
axes[1].set_xlabel('Diagnostic Class')
axes[1].set_ylabel('Patient Count')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Machine_Learning' / 'notebooks' / 'hd_target_dist.png', dpi=150)
plt.close()
print('Generated figure: hd_target_dist.png')


Multiclass distribution (num 0 to 4):
num
0    411
1    265
2    109
3    107
4     28
Name: count, dtype: int64

Binary CAD distribution (Presence vs Absence):
num
CAD Present (num>=1)    509
No CAD (num=0)          411
Name: count, dtype: int64


Generated figure: hd_target_dist.png


### 2.4 Numerical Feature Distributions & Outliers
Evaluating physiological parameters: `age`, `trestbps`, `chol`, `thalch`, and `oldpeak`.


In [6]:
num_cols_hd = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']
desc_hd = df_hd[num_cols_hd].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
print('Numerical Summary Statistics:')
print(desc_hd)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for i, col in enumerate(num_cols_hd):
    sns.boxplot(y=df_hd[col], ax=axes[i], color='lightblue')
    axes[i].set_title(col)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Machine_Learning' / 'notebooks' / 'hd_numerical_boxplots.png', dpi=150)
plt.close()
print('Generated figure: hd_numerical_boxplots.png')


Numerical Summary Statistics:
          count        mean         std   min    25%    50%    75%    max
age       920.0   53.510870    9.424685  28.0   47.0   54.0   60.0   77.0
trestbps  861.0  132.132404   19.066070   0.0  120.0  130.0  140.0  200.0
chol      890.0  199.130337  110.780810   0.0  175.0  223.0  268.0  603.0
thalch    865.0  137.545665   25.926276  60.0  120.0  140.0  157.0  202.0
oldpeak   858.0    0.878788    1.091226  -2.6    0.0    0.5    1.5    6.2


Generated figure: hd_numerical_boxplots.png


### 2.5 Categorical Features & Clinical Testing Sites
Evaluating distributions of `sex`, `dataset` (collection site), `cp` (chest pain type), `restecg`, `slope`, and `thal`.


In [7]:
cat_cols_hd = ['sex', 'dataset', 'cp', 'restecg', 'slope', 'thal']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(cat_cols_hd):
    ax = axes[i // 3, i % 3]
    df_hd[col].value_counts(dropna=False).plot(kind='bar', ax=ax, color='teal', edgecolor='black')
    ax.set_title(f'{col} (Missing: {df_hd[col].isnull().sum()})')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Machine_Learning' / 'notebooks' / 'hd_categorical_dist.png', dpi=150)
plt.close()
print('Generated figure: hd_categorical_dist.png')


Generated figure: hd_categorical_dist.png


### 2.6 Numerical Correlation Matrix
Linear correlations between patient vitals, ECG stress indicators, and target severity.


In [8]:
corr_hd = df_hd[num_cols_hd + ['num']].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr_hd, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('UCI Heart Disease: Numerical Feature Correlations')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Machine_Learning' / 'notebooks' / 'hd_correlation.png', dpi=150)
plt.close()
print('Generated figure: hd_correlation.png')


Generated figure: hd_correlation.png


### 2.7 Key Findings & Clinical Leakage Assessment (UCI Heart Disease)
1. **Hospital Site Protocol Disparities**: Fluoroscopy vessels (`ca`, 611 missing) and thallium scintigraphy (`thal`, 486 missing) were largely restricted to Cleveland patients. Models trained on non-Cleveland patients must avoid catastrophic missing-data artifacts.
2. **Physiological Zero Imputation**: The 172 records with `chol == 0` must be treated as `NaN` and imputed with median or model-based imputers fit strictly on training splits (Rule 9).
3. **Clinical Interpretation**: Maximum achieved heart rate (`thalch`) negatively correlates with CAD presence (-0.40), while ST depression (`oldpeak`) positively correlates (+0.40), reflecting established myocardial ischemia pathophysiology.


## 3. Heart Failure Clinical Records Dataset
### 3.1 Overview & Target Verification
- **Source**: Chicco & Jurman (2020), BMC Medical Informatics and Decision Making.
- **Population**: 299 patients with diagnosed heart failure (NYHA Class III-IV).
- **CRITICAL CLINICAL & ARCHITECTURAL VERIFICATION (Rule 8)**: The target is **`DEATH_EVENT`** (all-cause mortality during follow-up). This dataset does **NOT** diagnose heart failure. It stratifies survival/mortality outcome in confirmed heart failure patients.


In [9]:
df_hf = load_heart_failure_data()
print('Shape:', df_hf.shape)
print('Missing values total:', df_hf.isnull().sum().sum())
print('\nTarget DEATH_EVENT Distribution:')
print(df_hf['DEATH_EVENT'].value_counts())
print(f'Mortality rate in cohort: {round(df_hf["DEATH_EVENT"].mean() * 100, 2)}%')
df_hf.head()


2026-10-14 23:38:18,640 - Machine_Learning.src.data_loader - INFO - Loaded Heart Failure dataset: shape=(299, 13) from C:\Users\lodha\Downloads\Heart_Disease_Screening\Machine_Learning\datasets\raw\heart_failure.csv


Shape: (299, 13)
Missing values total: 0

Target DEATH_EVENT Distribution:
DEATH_EVENT
0    203
1     96
Name: count, dtype: int64
Mortality rate in cohort: 32.11%


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


### 3.2 Follow-up `time` & Look-Ahead Leakage Investigation
The feature `time` records the follow-up duration (in days) until death or study exit.
In prospective clinical intake, a clinician does not know how long the patient will be tracked in the future.
Conditioning on `time` produces look-ahead target leakage.


In [10]:
print('Follow-up Time (days) by Survival Status (DEATH_EVENT):')
print(df_hf.groupby('DEATH_EVENT')['time'].describe())

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=df_hf, x='time', hue='DEATH_EVENT', kde=True, bins=20, ax=ax, palette=['royalblue', 'crimson'])
ax.set_title('Follow-up Time Distribution by DEATH_EVENT (Evidence of Target Leakage)')
ax.set_xlabel('Follow-up Time (Days)')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Machine_Learning' / 'notebooks' / 'hf_time_leakage.png', dpi=150)
plt.close()
print('Generated figure: hf_time_leakage.png')


Follow-up Time (days) by Survival Status (DEATH_EVENT):
             count        mean        std   min   25%    50%     75%    max
DEATH_EVENT                                                                
0            203.0  158.339901  67.742872  12.0  95.0  172.0  213.00  285.0
1             96.0   70.885417  62.378281   4.0  25.5   44.5  102.25  241.0


Generated figure: hf_time_leakage.png


### 3.3 Clinical Biomarkers Analysis
Biomarkers: `ejection_fraction`, `serum_creatinine`, `serum_sodium`, `creatinine_phosphokinase`, and `platelets`.


In [11]:
biomarkers = ['ejection_fraction', 'serum_creatinine', 'serum_sodium', 'creatinine_phosphokinase', 'platelets']
print(df_hf[biomarkers].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
cols_to_plot = ['ejection_fraction', 'serum_creatinine', 'serum_sodium', 'creatinine_phosphokinase']
for i, col in enumerate(cols_to_plot):
    sns.boxplot(x='DEATH_EVENT', y=col, data=df_hf, ax=axes[i], palette=['skyblue', 'lightcoral'])
    axes[i].set_title(f'{col} vs DEATH_EVENT')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Machine_Learning' / 'notebooks' / 'hf_biomarkers.png', dpi=150)
plt.close()
print('Generated figure: hf_biomarkers.png')


2026-10-14 23:38:19,583 - matplotlib.category - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


C:\Users\lodha\AppData\Local\Temp\ipykernel_2396\1972076650.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='DEATH_EVENT', y=col, data=df_hf, ax=axes[i], palette=['skyblue', 'lightcoral'])
2026-10-14 23:38:19,601 - matplotlib.category - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


                          count           mean           std      min  \
ejection_fraction         299.0      38.083612     11.834841     14.0   
serum_creatinine          299.0       1.393880      1.034510      0.5   
serum_sodium              299.0     136.625418      4.412477    113.0   
creatinine_phosphokinase  299.0     581.839465    970.287881     23.0   
platelets                 299.0  263358.029264  97804.236869  25100.0   

                               50%       max  
ejection_fraction             38.0      80.0  
serum_creatinine               1.1       9.4  
serum_sodium                 137.0     148.0  
creatinine_phosphokinase     250.0    7861.0  
platelets                 262000.0  850000.0  


2026-10-14 23:38:19,668 - matplotlib.category - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


C:\Users\lodha\AppData\Local\Temp\ipykernel_2396\1972076650.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='DEATH_EVENT', y=col, data=df_hf, ax=axes[i], palette=['skyblue', 'lightcoral'])
2026-10-14 23:38:19,681 - matplotlib.category - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


2026-10-14 23:38:19,736 - matplotlib.category - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


C:\Users\lodha\AppData\Local\Temp\ipykernel_2396\1972076650.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='DEATH_EVENT', y=col, data=df_hf, ax=axes[i], palette=['skyblue', 'lightcoral'])
2026-10-14 23:38:19,749 - matplotlib.category - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


2026-10-14 23:38:19,806 - matplotlib.category - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


C:\Users\lodha\AppData\Local\Temp\ipykernel_2396\1972076650.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='DEATH_EVENT', y=col, data=df_hf, ax=axes[i], palette=['skyblue', 'lightcoral'])
2026-10-14 23:38:19,821 - matplotlib.category - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Generated figure: hf_biomarkers.png


### 3.4 Correlation Matrix (Heart Failure)
Evaluates pairwise correlations among biomarkers, demographics, `time`, and `DEATH_EVENT`.


In [12]:
corr_hf = df_hf.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr_hf, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Heart Failure Records: Correlation Heatmap')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Machine_Learning' / 'notebooks' / 'hf_correlation.png', dpi=150)
plt.close()
print('Generated figure: hf_correlation.png')


Generated figure: hf_correlation.png


### 3.5 Key Findings & Feasibility Guidelines (Heart Failure)
1. **Target Semantics**: The model predicts mortality hazard among patients with established heart failure. It must be presented strictly as "Heart Failure Follow-up Mortality Risk", never as "Heart Failure Diagnostic Probability".
2. **Leakage Elimination**: In prospective screening models, `time` must either be omitted from baseline predictors or modeled strictly through survival analysis horizons (e.g. 6-month or 1-year mortality probability).
3. **Core Predictive Biomarkers**: Ejection fraction (negatively correlated, r=-0.27) and serum creatinine (positively correlated, r=+0.29) carry the greatest physiological predictive power.


## 4. Arrhythmia Conditions Dataset
### 4.1 Overview & Schema Structure
- **Source**: UCI Arrhythmia Database (Guvenir et al.).
- **Shape**: 430 instances, 280 columns (features `'0'` to `'278'` and target `'condition_group'`).
- **Input Breakdown**:
  - Columns 0-14: Age, Sex, Height, Weight, ECG Intervals (QRS, P-R, Q-T, T, P), Vector Angles (QRS, T, P, QRST, J), Heart Rate.
  - Columns 15-278: 12-lead ECG waveform amplitudes, durations, deflections across 12 standard leads (DI, DII, DIII, AVR, AVL, AVF, V1, V2, V3, V4, V5, V6).


In [13]:
df_arr = load_arrhythmia_data()
print('Shape:', df_arr.shape)
target_counts = df_arr['condition_group'].value_counts()
print('\nTarget condition_group distribution:')
print(target_counts)

fig, ax = plt.subplots(figsize=(8, 4))
target_counts.plot(kind='barh', ax=ax, color='darkorchid', edgecolor='black')
ax.set_title('Arrhythmia Dataset: Condition Group Class Distribution')
ax.set_xlabel('Patient Count')
ax.set_ylabel('Condition Group')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Machine_Learning' / 'notebooks' / 'arrhythmia_classes.png', dpi=150)
plt.close()
print('Generated figure: arrhythmia_classes.png')


2026-10-14 23:38:24,122 - Machine_Learning.src.data_loader - INFO - Loaded Arrhythmia dataset: shape=(430, 280) from C:\Users\lodha\Downloads\Heart_Disease_Screening\Machine_Learning\datasets\raw\arrhythmia_conditions.csv


Shape: (430, 280)

Target condition_group distribution:
condition_group
Normal                           245
Ischemic_MI_Related               74
Conduction_Abnormality            59
Rhythm_Abnormality                43
Structural_Atrial_Abnormality      9
Name: count, dtype: int64


Generated figure: arrhythmia_classes.png


### 4.2 Missing Data Profiling
Evaluation of missingness across features.


In [14]:
arr_missing = df_arr.isnull().sum()
arr_missing_table = pd.DataFrame({
    'Feature': arr_missing[arr_missing > 0].index,
    'Missing Count': arr_missing[arr_missing > 0].values,
    'Missing %': (arr_missing[arr_missing > 0].values / len(df_arr) * 100).round(2)
})
print('Features with missing values:')
print(arr_missing_table.to_string(index=False))


Features with missing values:
Feature  Missing Count  Missing %
     10              7       1.63
     11             19       4.42
     12              1       0.23
     13            358      83.26
     14              1       0.23


### 4.3 Physical Outlier & Data Entry Anomaly Scan
Inspection of feature 2 (Height) reveals extreme values.


In [15]:
height_outliers = df_arr[df_arr['2'] > 250][['0', '1', '2', '3', 'condition_group']]
print('Extreme Height values (>250 cm):')
print(height_outliers)


Extreme Height values (>250 cm):
     0  1    2   3     condition_group
136  1  1  780   6  Rhythm_Abnormality
303  0  0  608  10  Rhythm_Abnormality


### 4.4 Critical Architectural Feasibility Assessment: 12-Lead ECG vs Outpatient Screening
- **Required Inputs**: The arrhythmia dataset requires **279 quantitative numerical features derived from 12-lead ECG waveforms** (QRS duration, intervals, vector angles, and lead deflections).
- **Clinical Intake Reality**: Outpatient screening questionnaires capture symptoms (palpitations, dizziness, chest fluttering) and vitals (resting heart rate, blood pressure). They do **NOT** provide 279 digitized ECG waveform deflection amplitudes.
- **Rule 7 & 20 Mandate**: The application must **NEVER** claim to predict arrhythmia condition groups from patient symptom questions or routine lab values without digitized 12-lead ECG data.
- **System Architecture Recommendation**:
  1. The arrhythmia prediction pipeline is designated for **ECG-Assisted Assessment mode**.
  2. In the absence of 12-lead ECG inputs, the system safely renders an **Explicit Unsupported Status** (`Status: PENDING_ECG`), routing the patient through symptom-based clinical safety triage rules instead of fabricating an ML output.


## 5. Cross-Dataset Synthesis & Modeling Guidelines
1. **Target Independence (Rule 11)**: The three datasets cannot be merged into a single arbitrary "total cardiovascular risk score". They address distinct medical tasks:
   - **Heart Disease UCI**: Diagnostic probability of coronary artery disease (>50% stenosis).
   - **Heart Failure Clinical**: Prognostic survival / mortality risk in known heart failure patients.
   - **Arrhythmia Conditions**: Electrophysiological classification of 12-lead ECG waveforms.
2. **Leakage Control**: Preprocessing pipelines must strictly fit all imputation, scaling, and encoding on training folds only.
3. **Phase 2 Status**: Data exploration, audit, and documentation are complete. Ready for pipeline implementation.
